In [2]:
import bw2data as bd

In [4]:
list(bd.projects)

[Project: default,
 Project: perocillo_cut_off_311,
 Project: cilleruelo_comparison_ei-3.8-cutoff,
 Project: multifunctional_cutoff,
 Project: default_25,
 Project: new_project_eiv2,
 Project: multifunctional_cutoff_bw25,
 Project: BAFU_2.1_zeyao,
 Project: premise_consequential_default,
 Project: pc_dup_bw-25_ecoinvent-3.12-apos_fm-fe,
 Project: pc_dup_bw-25_ecoinvent-3.12-cutoff_fm-fe,
 Project: ecoinvent-3.12-cutoff_edges,
 Project: motorforconf_SR-C_cutoff,
 Project: RA_ei-3.12_onlybw,
 Project: RA_ei-3.12_restored,
 Project: review_example,
 Project: material-composition,
 Project: CERC,
 Project: circularity_sensitivity_analysis,
 Project: brightcon,
 Project: CERC_3.12_consequential,
 Project: pc_dup_bw-25_ecoinvent-3.11-cutoff_fm-fe,
 Project: another_project,
 Project: pc_dup_bw-25_ecoinvent-3.12-consequential_fm-fe,
 Project: X,
 Project: Y,
 Project: libs_circularity_brightcon,
 Project: ecoinvent-3.12-consequential_fm-fe,
 Project: NAME]

In [5]:
bd.projects.set_current("NAME")

In [6]:
bd.databases

Databases dictionary with 3 object(s):
	bafu-2026
	bafu-2026-residual
	ef-3.1-biosphere

# Step 1: analyse aggregated datasets of a brightway project

# functions

In [7]:
from collections import defaultdict
import time
import os
import json
import logging
import bw2data as bd

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def find_aggregated_unit_processes(db_name):
    """
    Find aggregated unit processes in a database.
    Aggregated unit processes are defined as activities with:
    - Exactly one production exchange.
    - No technosphere exchanges.

    Args:
        db_name (str): Name of the database to analyze.

    Returns:
        list: List of activities that meet the criteria.
    """
    db = bd.Database(db_name)
    aggregated_unit_processes = []

    for act in db:
        # Check if the activity has exactly one production exchange
        production_exchanges = list(act.production())
        if len(production_exchanges) != 1:
            continue

        # Check if the activity has no technosphere exchanges
        technosphere_exchanges = list(act.technosphere())
        if len(technosphere_exchanges) > 0:
            continue

        aggregated_unit_processes.append(act)

    logger.info(f"Found {len(aggregated_unit_processes)} aggregated unit processes in database: {db_name}")
    for act in aggregated_unit_processes[:5]:
        logger.info(f"- {act.get('name')} ({act.get('location')}, {act.get('unit')})")
    if len(aggregated_unit_processes) > 5:
        logger.info("...")

    return aggregated_unit_processes

def analyze_all_databases_for_aggregated_unit_processes():
    """
    Analyze all databases for aggregated unit processes.

    Returns:
        dict: A dictionary with database names as keys and lists of aggregated unit processes as values.
    """
    logger.info("Analyzing all databases for aggregated unit processes...")
    all_aggregated_unit_processes = {}
    total = 0
    start = time.time()

    for db_name in bd.databases:
        processes = find_aggregated_unit_processes(db_name)
        all_aggregated_unit_processes[db_name] = processes
        total += len(processes)

    logger.info(f"Total aggregated unit processes found: {total}")
    logger.info(f"Analysis completed in {time.time() - start:.2f} s")

    return all_aggregated_unit_processes

def save_aggregated_unit_processes(results, filename=None):
    """
    Save aggregated unit processes to a JSON file.

    Args:
        results (dict): Dictionary of aggregated unit processes by database.
        filename (str, optional): Name of the JSON file. Defaults to "aggregated_unit_processes.json".
    """
    json_dir = os.path.join("results", "json")
    os.makedirs(json_dir, exist_ok=True)

    if filename is None:
        filename = "aggregated_unit_processes.json"

    filepath = os.path.join(json_dir, filename)

    with open(filepath, "w") as f:
        json.dump(
            {db_name: [act.as_dict() for act in acts]
             for db_name, acts in results.items()},
            f,
            indent=2
        )
    logger.info(f"Aggregated unit processes saved to: {filepath}")

## exe

In [8]:
# Analyze all databases for aggregated unit processes
aggregated_processes = analyze_all_databases_for_aggregated_unit_processes()

# Save the results to a JSON file
save_aggregated_unit_processes(aggregated_processes)

INFO:__main__:Analyzing all databases for aggregated unit processes...
INFO:__main__:Found 0 aggregated unit processes in database: ef-3.1-biosphere
INFO:__main__:Found 0 aggregated unit processes in database: bafu-2026-residual
INFO:__main__:Found 276 aggregated unit processes in database: bafu-2026
INFO:__main__:- Titanium dioxide, chloride process, at plant (RER, kilogram)
INFO:__main__:- Waste Cooking Oil (RER, megajoule)
INFO:__main__:- xx MnO2 powder, pyrometallurgical processing Li-ion batteries, at plant (GLO, kilogram)
INFO:__main__:- xx Al fraction, mechanical treatment, printer, laser, at plant (GLO, kilogram)
INFO:__main__:- Disposal, sheet pile wall, strutted apart, vibrated, Schulhaus Sandgruben Basel (CH, square meter)
INFO:__main__:...
INFO:__main__:Total aggregated unit processes found: 276
INFO:__main__:Analysis completed in 71.42 s
INFO:__main__:Aggregated unit processes saved to: results\json\aggregated_unit_processes.json
